# JWT authentication and API testing

Build the authentication boundary, then attack it with tests.

In [ ]:
# %pip install fastapi pyjwt passlib[bcrypt] httpx pytest
from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
import jwt, os

app = FastAPI()
security = HTTPBearer()
SECRET = os.environ.get('JWT_SECRET', 'development-only-secret')

def issue_token(user_id: str, role: str):
    return jwt.encode({'sub': user_id, 'role': role}, SECRET, algorithm='HS256')

In [ ]:
def current_user(credentials: HTTPAuthorizationCredentials = Depends(security)):
    try:
        return jwt.decode(credentials.credentials, SECRET, algorithms=['HS256'])
    except jwt.InvalidTokenError:
        raise HTTPException(status_code=401, detail='invalid_token')

@app.get('/v1/me')
def me(user=Depends(current_user)):
    return user

In [ ]:
from fastapi.testclient import TestClient
client = TestClient(app)
token = issue_token('user-123', 'member')
response = client.get('/v1/me', headers={'Authorization': f'Bearer {token}'})
assert response.status_code == 200
assert response.json()['sub'] == 'user-123'

## Exercises

1. Add `exp`, `iss`, and `aud` claims and validate them.
2. Implement refresh-token rotation and revocation.
3. Add role-based authorization.
4. Add object-level authorization for project IDs.
5. Test expired, malformed, revoked, and wrong-audience tokens.
6. Move the secret to production secret management.
7. Add rate limiting to login and refresh endpoints.
8. Never log bearer tokens in test or application output.